# 08 -- Full ablation report

Legacy ablation report for the original architecture rows. For paper-facing comparisons, use notebook 14, which adds `matched_dense` total/active/MAC controls, both baseline curricula, immutable shared Stage-A checkpoints, `Resource_Accounting.csv`, and paired seeds 0/1/2. Do not merge this notebook's historical cold-start rows into the revised fairness table.


## Setup

Run this cell first. It's the ONLY cell you should need to edit: change
`CONFIG_OVERRIDES` (a list of `--set key.path=value` style dotted overrides,
same syntax as `training.run`'s CLI) to narrow `data.active_datasets`,
switch `architecture`, point at a different Drive folder, etc.


In [ ]:
# ---- Single config cell: this is the only cell you should need to edit ----
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess, os
    REPO_DIR = '/content/dataset_moe_nids'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/selimsidan/dataset_moe_nids', REPO_DIR], check=True)
    %cd $REPO_DIR
    %pip install -q -r requirements.txt

CONFIG_PATH = 'config/default.yaml'

# Dotted --set overrides, same syntax as training.run's CLI. Narrow
# data.active_datasets to 2-3 datasets here for a fast iteration cycle --
# every downstream module (registry, harmonizer, expert-bank sizing,
# checkpoints) adapts automatically, no other code changes needed.
CONFIG_OVERRIDES = [
    # 'data.active_datasets=[NF-UNSW-NB15-v3,NF-BoT-IoT-v3]',
    # 'architecture=moe_dataset_soft',
    # 'training.device=cuda',
]

from training.config import load_config
config = load_config(CONFIG_PATH, CONFIG_OVERRIDES)
print('run_name:', config['run_name'])
print('architecture:', config['architecture'])
print('active_datasets:', config['data']['active_datasets'])
print('checkpoint_dir:', config['training']['checkpoint_dir'])


In [ ]:
import copy
import uuid
import torch
import torch.nn.functional as F
import numpy as np

from training.dataset import prepare_datasets
from training.stage_a_pretrain import run_stage_a
from training.stage_b_warmstart import run_stage_b
from training.stage_c_jointfinetune import run_stage_c, build_model_from_checkpoints
from training.baseline_train import train_hard_two_stage, train_no_fusion, train_plain_pooled
from training.checkpoint import load_stage_c, save_harmonizer, stage_complete
from training.run import MOE_ARCHITECTURES
from training.config import apply_architecture_defaults
from evaluation.metrics import evaluate_per_dataset, evaluate_predictions
from evaluation.bootstrap_ci import bootstrap_per_class_ci
from evaluation.report import write_tracker_csvs, comparison_table, per_dataset_comparison_table

data = prepare_datasets(config)
device = torch.device(config['training'].get('device', 'cpu'))
results, per_dataset_results, ci_results = {}, {}, {}

def _predict(model, dataset_names=None):
    model.eval()
    if dataset_names is None:
        with torch.no_grad():
            output = model(torch.from_numpy(data.test.features))
            scores = output['combined_probs'] if 'combined_probs' in output else F.softmax(output['logits'], dim=1)
            scores = scores.numpy(); preds = scores.argmax(axis=1)
        return data.test.class_idx, preds, data.test.dataset_name, scores
    y_true_l, y_pred_l, ds_l, score_l = [], [], [], []
    with torch.no_grad():
        for name in dataset_names:
            mask = data.test.dataset_name == name
            if mask.sum() == 0:
                continue
            scores = F.softmax(model(torch.from_numpy(data.test.features[mask]), name)['logits'], dim=1).numpy()
            y_true_l.append(data.test.class_idx[mask]); y_pred_l.append(scores.argmax(axis=1)); ds_l.append(data.test.dataset_name[mask]); score_l.append(scores)
    return np.concatenate(y_true_l), np.concatenate(y_pred_l), np.concatenate(ds_l), np.concatenate(score_l)

for arch in MOE_ARCHITECTURES:
    variant_cfg = copy.deepcopy(config)
    variant_cfg['architecture'] = arch
    apply_architecture_defaults(variant_cfg)
    variant_cfg['training']['checkpoint_dir'] = config['training']['checkpoint_dir'] + f'_{arch}'
    save_harmonizer(variant_cfg['training']['checkpoint_dir'], data.harmonizer)
    if not stage_complete(variant_cfg['training']['checkpoint_dir'], 'A'):
        run_stage_a(variant_cfg, data)
    if not stage_complete(variant_cfg['training']['checkpoint_dir'], 'B'):
        run_stage_b(variant_cfg, data)
    if not stage_complete(variant_cfg['training']['checkpoint_dir'], 'C'):
        run_stage_c(variant_cfg, data)
    model = build_model_from_checkpoints(variant_cfg, data, device)
    model.load_state_dict(load_stage_c(variant_cfg['training']['checkpoint_dir'])['model_state'])
    y_true, preds, ds_names, scores = _predict(model)
    results[arch] = evaluate_predictions(y_true, preds, data.class_names, scores)
    per_dataset_results[arch] = evaluate_per_dataset(y_true, preds, ds_names, data.class_names, scores)
    ci_results[arch] = bootstrap_per_class_ci(y_true, preds, data.class_names,
        low_sample_threshold=config['evaluation']['low_sample_threshold'],
        n_bootstrap=config['evaluation']['bootstrap_n'], seed=config['evaluation']['bootstrap_seed'])

for name, train_fn, needs_ds in [
    ('plain_pooled', train_plain_pooled, False),
    ('no_fusion', train_no_fusion, True),
    ('hard_two_stage', train_hard_two_stage, False),
]:
    baseline_cfg = copy.deepcopy(config)
    baseline_cfg['architecture'] = name
    model = train_fn(baseline_cfg, data)
    y_true, preds, ds_names, scores = _predict(model, dataset_names=data.active_datasets if needs_ds else None)
    results[name] = evaluate_predictions(y_true, preds, data.class_names, scores)
    per_dataset_results[name] = evaluate_per_dataset(y_true, preds, ds_names, data.class_names, scores)
    ci_results[name] = bootstrap_per_class_ci(y_true, preds, data.class_names,
        low_sample_threshold=config['evaluation']['low_sample_threshold'],
        n_bootstrap=config['evaluation']['bootstrap_n'], seed=config['evaluation']['bootstrap_seed'])

trial_id = f"{config['run_name']}-ablation-{uuid.uuid4().hex[:8]}"
write_tracker_csvs(config['evaluation']['output_dir'], trial_id, config, results, ci_results, per_dataset_results)
print('Wrote tracker CSVs to', config['evaluation']['output_dir'])

per_dataset_comparison_table(per_dataset_results)


In [ ]:
comparison_table(results)
